# Chapter 5 — Resource-audit results summary and publication exports

This notebook reads the outputs produced by the final QA1, QA2, QA3, and ControlledStress notebooks. It does **not** recompute the audit metrics. Instead, it validates the exported schemas, reconstructs the chapter-level summaries from the current metric definitions, and exports publication-ready PDF/PNG figures, CSV/LaTeX tables, key-result JSON, LaTeX macros, and a reproducibility manifest.

The previous chapter draft must be updated from these exports rather than from older hard-coded values. In particular:

- QA1.2 is reported by target language and component class.
- QA2.2 reports coverage and groundedness separately.
- QA2 review localisation uses only the final defined review decisions and exact literal checks.
- QA3.2 is descriptive and is not included in the aggregate corruption-detection rate.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import warnings
from pathlib import Path
from textwrap import fill

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

LANGUAGE_ORDER = ["Spanish", "Catalan"]
EXPECTED_CORE_COMPATIBILITY = "audit-text-aligned-2.7"

def find_repo_root() -> Path:
    override = os.getenv("WEBNLG_REPO_ROOT")
    if override:
        candidate = Path(override).expanduser().resolve()
        if not candidate.exists():
            raise FileNotFoundError(
                f"WEBNLG_REPO_ROOT does not exist: {candidate}"
            )
        return candidate

    for start in [Path.cwd().resolve(), Path.home().resolve()]:
        for candidate in [start, *start.parents]:
            if (
                (candidate / "WebNLG_CA_BT").exists()
                or (candidate / "audit" / "QA_evaluation").exists()
                or (candidate / "QA_evaluation").exists()
            ):
                return candidate

    raise FileNotFoundError(
        "Repository root not found. Set WEBNLG_REPO_ROOT before running "
        "this notebook."
    )

REPO_ROOT = find_repo_root()

def resolve_results_dir(
    env_name: str,
    result_name: str,
    sentinel: str,
) -> Path:
    override = os.getenv(env_name)
    if override:
        return Path(override).expanduser().resolve()

    candidates = [
        REPO_ROOT / "audit" / "QA_evaluation" / "results" / result_name,
        REPO_ROOT / "QA_evaluation" / "results" / result_name,
    ]
    for candidate in candidates:
        if (candidate / sentinel).exists():
            return candidate
    return candidates[0]

QA1_RESULTS_DIR = resolve_results_dir(
    "QA1_RESULTS_DIR",
    os.getenv("QA1_RESULTS_NAME", "QA1_stress_aligned"),
    "qa11_original_structural_summary.csv",
)
QA2_RESULTS_DIR = resolve_results_dir(
    "QA2_RESULTS_DIR",
    os.getenv("QA2_RESULTS_NAME", "QA2_stress_aligned"),
    "qa2_original_summary.csv",
)
QA3_RESULTS_DIR = resolve_results_dir(
    "QA3_RESULTS_DIR",
    os.getenv("QA3_RESULTS_NAME", "QA3_stress_aligned"),
    "qa3_original_language_summary.csv",
)
STRESS_RESULTS_DIR = resolve_results_dir(
    "CONTROLLEDSTRESS_RESULTS_DIR",
    os.getenv("ControlledStress_RESULTS_NAME", "ControlledStress_final_audit"),
    "combined_detection_summary.csv",
)

EXPORT_DIR = Path(
    os.getenv(
        "CHAPTER_FRAMEWORK_EXPORT_DIR",
        str(
            REPO_ROOT
            "..."
        ),
    )
).expanduser().resolve()
FIGURE_DIR = EXPORT_DIR / "figures"
TABLE_DIR = EXPORT_DIR / "tables"
DATA_DIR = EXPORT_DIR / "data"
for directory in [EXPORT_DIR, FIGURE_DIR, TABLE_DIR, DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("QA1 results:", QA1_RESULTS_DIR)
print("QA2 results:", QA2_RESULTS_DIR)
print("QA3 results:", QA3_RESULTS_DIR)
print("ControlledStress results:", STRESS_RESULTS_DIR)
print("Chapter exports:", EXPORT_DIR)


Repository root: /home/vramon/notebooks/WebNLG_MT_eval/audit
QA1 results: /home/vramon/notebooks/WebNLG_MT_eval/audit/audit/QA_evaluation/results/QA1_stress_aligned
QA2 results: /home/vramon/notebooks/WebNLG_MT_eval/audit/audit/QA_evaluation/results/QA2_stress_aligned
QA3 results: /home/vramon/notebooks/WebNLG_MT_eval/audit/audit/QA_evaluation/results/QA3_stress_aligned
AuditStress results: /home/vramon/notebooks/WebNLG_MT_eval/audit/QA_evaluation/results/AuditStress_final_audit
Chapter exports: /home/vramon/notebooks/WebNLG_MT_eval/audit/audit/QA_evaluation/results/Chapter5_audit_summary_aligned


## 1. Load and validate the final notebook outputs

The summary is intentionally based on exported results rather than notebook memory. This makes it possible to rerun the four metric notebooks independently and then regenerate every chapter table and figure without repeating model inference.


In [2]:
def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Required result file not found: {path}\n"
            "Run the corresponding final metric notebook first, or set the "
            "appropriate *_RESULTS_DIR environment variable."
        )
    return path


def load_csv(directory: Path, filename: str, required: bool = True) -> pd.DataFrame:
    path = directory / filename
    if not path.exists():
        if required:
            require_file(path)
        return pd.DataFrame()
    frame = pd.read_csv(path)
    print(f"Loaded {path}: {len(frame):,} rows")
    return frame


def load_json(directory: Path, filename: str, required: bool = False) -> dict:
    path = directory / filename
    if not path.exists():
        if required:
            require_file(path)
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def load_pickle_or_csv(
    directory: Path,
    pickle_name: str,
    csv_name: str,
) -> pd.DataFrame:
    pickle_path = directory / pickle_name
    if pickle_path.exists():
        frame = pd.read_pickle(pickle_path)
        print(f"Loaded {pickle_path}: {len(frame):,} rows")
        return frame
    return load_csv(directory, csv_name, required=True)


def require_columns(frame: pd.DataFrame, columns: list[str], label: str) -> None:
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def coerce_boolean(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float).ne(0)
    normalized = series.fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"true", "1", "yes", "y", "t"})


def finite_range_check(
    frame: pd.DataFrame,
    columns: list[str],
    lower: float,
    upper: float,
    label: str,
) -> None:
    for column in columns:
        if column not in frame:
            continue
        values = pd.to_numeric(frame[column], errors="coerce").dropna()
        if not values.between(lower, upper, inclusive="both").all():
            raise AssertionError(
                f"{label}.{column} must lie in [{lower}, {upper}]. "
                f"Observed [{values.min()}, {values.max()}]."
            )


def file_fingerprint(path: Path) -> dict:
    if not path.exists():
        return {"path": str(path), "exists": False}
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return {
        "path": str(path),
        "exists": True,
        "bytes": path.stat().st_size,
        "sha256": digest.hexdigest(),
    }


In [ ]:
# Original-resource outputs
qa11_summary = load_csv(
    QA1_RESULTS_DIR, "qa11_original_structural_summary.csv"
)
qa12_summary = load_csv(
    QA1_RESULTS_DIR, "qa12_original_consistency_summary.csv"
)
qa12_variable = load_csv(
    QA1_RESULTS_DIR, "qa12_variable_mappings.csv"
)
qa1_inventory = load_csv(
    QA1_RESULTS_DIR, "qa1_entry_inventory.csv"
)
qa1_metadata = load_json(
    QA1_RESULTS_DIR, "qa1_run_metadata.json"
)

qa2_summary = load_csv(
    QA2_RESULTS_DIR, "qa2_original_summary.csv"
)
qa2_metrics = load_pickle_or_csv(
    QA2_RESULTS_DIR,
    "qa2_lexicalisation_metrics.pkl",
    "qa2_lexicalisation_metrics.csv",
)
qa2_thresholds = load_csv(
    QA2_RESULTS_DIR, "qa22_english_calibration_thresholds.csv"
)
qa2_bootstrap = load_csv(
    QA2_RESULTS_DIR, "qa2_spanish_catalan_cluster_bootstrap.csv"
)
qa2_metadata = load_json(
    QA2_RESULTS_DIR, "qa2_run_metadata.json"
)

qa3_language_summary = load_csv(
    QA3_RESULTS_DIR, "qa3_original_language_summary.csv"
)
qa3_variation_summary = load_csv(
    QA3_RESULTS_DIR, "qa3_variation_summary.csv"
)
qa3_metrics = load_pickle_or_csv(
    QA3_RESULTS_DIR,
    "qa3_lexicalisation_metrics.pkl",
    "qa3_lexicalisation_metrics.csv",
)
qa3_metadata = load_json(
    QA3_RESULTS_DIR, "qa3_run_metadata.json"
)

# Controlled stress-test outputs
stress_combined = load_csv(
    STRESS_RESULTS_DIR, "combined_detection_summary.csv"
)
stress_qa1_structural = load_csv(
    STRESS_RESULTS_DIR, "qa1_structural_detection_summary.csv"
)
stress_qa1_consistency = load_csv(
    STRESS_RESULTS_DIR, "qa1_consistency_selectivity_summary.csv"
)
stress_qa2 = load_csv(
    STRESS_RESULTS_DIR, "qa2_corruption_detection_summary.csv"
)
stress_qa2_benign = load_csv(
    STRESS_RESULTS_DIR, "qa2_benign_variation_summary.csv"
)
stress_qa2_omission = load_csv(
    STRESS_RESULTS_DIR,
    "qa2_omission_severity_summary.csv",
    required=False,
)
stress_qa2_addition = load_csv(
    STRESS_RESULTS_DIR,
    "qa2_addition_severity_summary.csv",
    required=False,
)
stress_qa3 = load_csv(
    STRESS_RESULTS_DIR, "qa3_corruption_detection_summary.csv"
)
stress_qa3_controls = load_csv(
    STRESS_RESULTS_DIR, "qa3_control_selectivity_summary.csv"
)
stress_qa3_benign = load_csv(
    STRESS_RESULTS_DIR, "qa3_benign_variation_summary.csv"
)
stress_metadata = load_json(
    STRESS_RESULTS_DIR, "ControlledStress_run_metadata.json"
)


FileNotFoundError: Required result file not found: /home/vramon/notebooks/WebNLG_MT_eval/audit/audit/QA_evaluation/results/QA1_stress_aligned/qa11_original_structural_summary.csv
Run the corresponding final metric notebook first, or set the appropriate *_RESULTS_DIR environment variable.

In [ ]:
require_columns(
    qa11_summary,
    [
        "language", "language_label", "rdf_entries",
        "integrity_pass_rate", "target_triple_parity_rate",
        "target_text_present_rate", "lexicalisation_id_alignment_rate",
        "lexicalisation_id_unique_rate", "target_rdf_well_formed_rate",
    ],
    "QA1.1 summary",
)
require_columns(
    qa12_summary,
    [
        "language", "language_label", "component_type",
        "recurring_source_items", "mapping_observations",
        "weighted_dominant_mapping_rate", "mapping_review_rate",
        "items_selected_for_review",
    ],
    "QA1.2 summary",
)
require_columns(
    qa2_summary,
    [
        "language", "language_label", "lexicalisations", "records",
        "text_similarity_mean", "triple_similarity_mean",
        "predicate_similarity_mean",
        "source_target_full_literal_preservation_rate",
        "minimum_triple_coverage_mean", "coverage_pass_rate",
        "minimum_text_groundedness_mean", "groundedness_pass_rate",
        "target_text_full_literal_retention_rate",
        "relative_literal_retention",
    ],
    "QA2 summary",
)
require_columns(
    qa2_metrics,
    [
        "language", "language_label",
        "structured_predicate_review_flag",
        "source_target_literal_preservation",
        "english_calibrated_coverage_pass",
        "english_calibrated_groundedness_pass",
        "target_text_literal_retention",
    ],
    "QA2 instance metrics",
)
require_columns(
    qa3_language_summary,
    [
        "language", "language_label", "variants",
        "hard_language_flag_rate", "valid_target_language_rate",
        "wrong_language_rate", "exact_source_copy_rate",
        "code_switch_rate", "mean_target_language_probability",
    ],
    "QA3.1 summary",
)
require_columns(
    qa3_variation_summary,
    [
        "language", "language_label", "mean_expansion_ratio",
        "median_expansion_ratio", "expansion_ratio_sd",
        "aligned_reference_eligible",
        "mean_aligned_reference_advantage",
        "positive_aligned_reference_advantage_rate",
    ],
    "QA3.2 summary",
)
require_columns(
    qa3_metrics,
    [
        "language", "language_label", "expansion_ratio",
        "aligned_reference_advantage",
    ],
    "QA3 instance metrics",
)
require_columns(
    stress_combined,
    [
        "audit_dimension", "language", "language_label",
        "corruption_type", "severity", "variants",
        "primary_detection_rate",
    ],
    "ControlledStress combined summary",
)
require_columns(
    stress_qa2,
    [
        "language", "language_label", "corruption_type", "severity",
        "variants", "primary_metric", "expected_direction_rate",
        "mean_delta", "cluster_bootstrap_ci_low",
        "cluster_bootstrap_ci_high",
    ],
    "ControlledStress QA2 summary",
)
require_columns(
    stress_qa3_controls,
    [
        "qa_track", "language", "language_label", "variants",
        "hard_flag_false_positive_rate", "valid_target_language_rate",
        "wrong_language_rate", "exact_copy_rate", "code_switch_rate",
    ],
    "ControlledStress QA3 controls",
)

finite_range_check(
    qa11_summary,
    [
        "integrity_pass_rate", "target_triple_parity_rate",
        "target_text_present_rate", "lexicalisation_id_alignment_rate",
        "lexicalisation_id_unique_rate", "target_rdf_well_formed_rate",
    ],
    0.0, 1.0, "QA1.1",
)
finite_range_check(
    qa12_summary,
    ["weighted_dominant_mapping_rate", "mapping_review_rate"],
    0.0, 1.0, "QA1.2",
)
finite_range_check(
    qa2_summary,
    [
        "coverage_pass_rate", "groundedness_pass_rate",
        "source_target_full_literal_preservation_rate",
        "target_text_full_literal_retention_rate",
        "relative_literal_retention",
    ],
    0.0, 1.0, "QA2",
)
finite_range_check(
    qa3_language_summary,
    [
        "hard_language_flag_rate", "valid_target_language_rate",
        "wrong_language_rate", "exact_source_copy_rate",
        "code_switch_rate", "mean_target_language_probability",
    ],
    0.0, 1.0, "QA3.1",
)
finite_range_check(
    stress_combined,
    ["primary_detection_rate"],
    0.0, 1.0, "ControlledStress",
)

if (pd.to_numeric(qa3_metrics["expansion_ratio"], errors="coerce") < 0).any():
    raise AssertionError("QA3.2 expansion ratios must be non-negative.")

metadata_versions = {
    "QA1": qa1_metadata.get("metric_core_compatibility"),
    "QA2": qa2_metadata.get("metric_core_compatibility"),
    "QA3": qa3_metadata.get("metric_core_compatibility"),
}
for name, version in metadata_versions.items():
    if version and version != EXPECTED_CORE_COMPATIBILITY:
        warnings.warn(
            f"{name} outputs report metric-core compatibility {version!r}; "
            f"the final shared core is {EXPECTED_CORE_COMPATIBILITY!r}. "
            "Rerun the compatibility-corrected notebook for unified provenance."
        )

print("All required result schemas passed validation.")


## 2. Publication-export helpers


In [ ]:
CORRUPTION_LABELS = {
    "qa1_delete_triple": "Deleted triple",
    "qa1_duplicate_triple": "Duplicated triple",
    "qa1_missing_text": "Missing text",
    "qa1_wrong_lid": "Wrong lexicalisation ID",
    "qa1_duplicate_lid": "Duplicated lexicalisation ID",
    "qa1_placeholder_component": "Placeholder component",
    "qa1_missing_component": "Missing RDF component",
    "qa1_source_markup": "Residual source markup",
    "qa1_mapping_inconsistency": "Mapping inconsistency",
    "qa2_wrong_record_text": "Wrong-record text",
    "qa2_entity_substitution_triple": "Entity substitution in triples",
    "qa2_predicate_substitution_triple": "Predicate substitution",
    "qa2_structured_predicate_retention": "Structured unchanged-predicate challenge",
    "qa2_entity_substitution_text": "Entity substitution in text",
    "qa2_literal_text_only": "Literal changed in text",
    "qa2_literal_both": "Literal changed in triples and text",
    "qa2_omit_one_sentence": "Mild omission",
    "qa2_omit_to_first_sentence": "Severe omission",
    "qa2_unsupported_addition": "Unsupported addition",
    "qa2_unsupported_addition_plausible": "Plausible unsupported addition",
    "qa2_unsupported_addition_unrelated": "Unrelated unsupported addition",
    "qa3_full_english_copy": "Full English copy",
    "qa3_english_clause_insertion": "English-clause insertion",
}
METRIC_LABELS = {
    "triple_coverage": "Minimum triple coverage",
    "text_groundedness": "Minimum text groundedness",
    "triple_similarity": "Triple-set similarity",
    "text_similarity": "Source-to-target text similarity",
    "literal_retention": "Target triple-to-text literal retention",
    "source_target_literal_preservation": "Source-to-target literal preservation",
    "predicate_similarity": "Predicate similarity",
    "structured_predicate_review_flag": "Structured predicate review flag",
}
COMPONENT_LABELS = {
    "entity_or_value": "Entity or value",
    "predicate": "Predicate",
}

def pretty_corruption(value: object) -> str:
    return CORRUPTION_LABELS.get(
        str(value),
        str(value).replace("_", " ").title(),
    )

def pretty_metric(value: object) -> str:
    return METRIC_LABELS.get(
        str(value),
        str(value).replace("_", " ").title(),
    )

def save_figure(fig, stem: str) -> None:
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{stem}.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)

def export_table(
    frame: pd.DataFrame,
    stem: str,
    caption: str,
    label: str,
    *,
    latex_frame: pd.DataFrame | None = None,
    float_format: str = "%.3f",
) -> None:
    frame.to_csv(TABLE_DIR / f"{stem}.csv", index=False)
    display_frame = frame if latex_frame is None else latex_frame
    latex = display_frame.to_latex(
        index=False,
        escape=True,
        float_format=float_format,
        caption=caption,
        label=label,
        position="htbp",
    )
    (TABLE_DIR / f"{stem}.tex").write_text(
        latex,
        encoding="utf-8",
    )

def language_reindex(frame: pd.DataFrame) -> pd.DataFrame:
    if "language_label" in frame.columns:
        return (
            frame.assign(
                language_label=pd.Categorical(
                    frame["language_label"],
                    categories=LANGUAGE_ORDER,
                    ordered=True,
                )
            )
            .sort_values("language_label")
            .reset_index(drop=True)
        )
    return frame

def format_measure(value: object, unit: str) -> str:
    if pd.isna(value):
        return "--"
    numeric = float(value)
    if unit == "count":
        return f"{int(round(numeric)):,}"
    if unit == "rate5":
        return f"{numeric:.5f}"
    return f"{numeric:.3f}"


## 3. Original-resource audit


In [ ]:
inventory = (
    qa11_summary[
        ["language", "language_label", "rdf_entries"]
    ]
    .merge(
        qa2_summary[["language", "lexicalisations"]],
        on="language",
        how="inner",
        validate="one_to_one",
    )
    .rename(columns={
        "language_label": "Language",
        "rdf_entries": "RDF entries",
        "lexicalisations": "Aligned lexicalisations",
    })
)
inventory = language_reindex(
    inventory.rename(columns={"Language": "language_label"})
).rename(columns={"language_label": "Language"})
inventory = inventory[
    ["Language", "RDF entries", "Aligned lexicalisations"]
]
inventory["RDF entries"] = inventory["RDF entries"].astype(int)
inventory["Aligned lexicalisations"] = inventory[
    "Aligned lexicalisations"
].astype(int)

display(inventory)
inventory_tex = inventory.copy()
for column in ["RDF entries", "Aligned lexicalisations"]:
    inventory_tex[column] = inventory_tex[column].map(lambda value: f"{value:,}")
export_table(
    inventory,
    "table_original_resource_inventory",
    "Inventory of aligned RDF entries and lexicalisations in the original-resource audit.",
    "tab:original-resource-inventory",
    latex_frame=inventory_tex,
)


In [ ]:
qa11_by_language = qa11_summary.set_index("language_label")
qa2_by_language = qa2_summary.set_index("language_label")
qa3_language_by_language = qa3_language_summary.set_index("language_label")
qa3_variation_by_language = qa3_variation_summary.set_index("language_label")

overview_rows = []

def append_overview(
    qa: str,
    indicator: str,
    source: pd.DataFrame,
    column: str,
    unit: str,
) -> None:
    overview_rows.append({
        "QA": qa,
        "Indicator": indicator,
        "Spanish": source.loc["Spanish", column],
        "Catalan": source.loc["Catalan", column],
        "Unit": unit,
    })

append_overview(
    "QA1.1", "Record integrity rate",
    qa11_by_language, "integrity_pass_rate", "rate",
)

for component_type, component_label in COMPONENT_LABELS.items():
    subset = qa12_summary[
        qa12_summary["component_type"].eq(component_type)
    ].set_index("language_label")
    append_overview(
        "QA1.2",
        f"Weighted dominant mapping rate — {component_label.lower()}",
        subset,
        "weighted_dominant_mapping_rate",
        "rate5",
    )
    append_overview(
        "QA1.2",
        f"Recurring mappings selected for review — {component_label.lower()}",
        subset,
        "items_selected_for_review",
        "count",
    )

for qa, indicator, column, unit in [
    ("QA2.1", "Verbalisation similarity (TextSim)", "text_similarity_mean", "similarity"),
    ("QA2.1", "Triple-set similarity (TripleSim)", "triple_similarity_mean", "similarity"),
    ("QA2.1", "Predicate similarity (PredSim)", "predicate_similarity_mean", "similarity"),
    ("QA2.1", "Full source-to-target literal preservation rate", "source_target_full_literal_preservation_rate", "rate"),
    ("QA2.2", "English-calibrated coverage pass rate", "coverage_pass_rate", "rate"),
    ("QA2.2", "English-calibrated groundedness pass rate", "groundedness_pass_rate", "rate"),
    ("QA2.2", "Relative literal retention", "relative_literal_retention", "rate"),
]:
    append_overview(qa, indicator, qa2_by_language, column, unit)

for qa, indicator, column, unit in [
    ("QA3.1", "Valid target-language rate", "valid_target_language_rate", "rate"),
    ("QA3.1", "High-confidence wrong-language rate", "wrong_language_rate", "rate5"),
    ("QA3.1", "Exact English-copy rate", "exact_source_copy_rate", "rate5"),
    ("QA3.1", "English code-switch rate", "code_switch_rate", "rate5"),
]:
    append_overview(qa, indicator, qa3_language_by_language, column, unit)

append_overview(
    "QA3.2", "Mean expansion ratio",
    qa3_variation_by_language, "mean_expansion_ratio", "ratio",
)
append_overview(
    "QA3.2", "Positive aligned-reference advantage rate",
    qa3_variation_by_language,
    "positive_aligned_reference_advantage_rate",
    "rate",
)

original_overview = pd.DataFrame(overview_rows)
display(original_overview)

original_overview_tex = original_overview.copy()
for language in LANGUAGE_ORDER:
    original_overview_tex[language] = [
        format_measure(value, unit)
        for value, unit in zip(
            original_overview_tex[language],
            original_overview_tex["Unit"],
        )
    ]
original_overview_tex = original_overview_tex.drop(columns="Unit")

export_table(
    original_overview,
    "table_original_audit_summary",
    "Summary of the original-resource audit under the final metric definitions.",
    "tab:original-audit-summary",
    latex_frame=original_overview_tex,
)


### 3.1 QA1 structural integrity and recurring-component consistency


In [ ]:
qa11_detail = language_reindex(qa11_summary.copy())[
    [
        "language_label",
        "rdf_entries",
        "integrity_pass_rate",
        "target_triple_parity_rate",
        "target_text_present_rate",
        "lexicalisation_id_alignment_rate",
        "lexicalisation_id_unique_rate",
        "target_rdf_well_formed_rate",
    ]
].rename(columns={
    "language_label": "Language",
    "rdf_entries": "RDF entries",
    "integrity_pass_rate": "Record integrity",
    "target_triple_parity_rate": "Triple parity",
    "target_text_present_rate": "Text present",
    "lexicalisation_id_alignment_rate": "LID alignment",
    "lexicalisation_id_unique_rate": "LID uniqueness",
    "target_rdf_well_formed_rate": "RDF well-formedness",
})
display(qa11_detail.round(6))
export_table(
    qa11_detail,
    "table_original_qa11_details",
    "QA1.1 structural-integrity indicators on the original resources.",
    "tab:original-qa11-details",
)

qa12_detail = qa12_summary.copy()
qa12_detail["component_type"] = qa12_detail[
    "component_type"
].map(COMPONENT_LABELS).fillna(qa12_detail["component_type"])
qa12_detail = language_reindex(qa12_detail)[
    [
        "language_label",
        "component_type",
        "recurring_source_items",
        "mapping_observations",
        "weighted_dominant_mapping_rate",
        "mapping_review_rate",
        "items_selected_for_review",
    ]
].rename(columns={
    "language_label": "Language",
    "component_type": "Component class",
    "recurring_source_items": "Recurring source items",
    "mapping_observations": "Mapping observations",
    "weighted_dominant_mapping_rate": "Weighted dominant mapping rate",
    "mapping_review_rate": "Review-signal rate",
    "items_selected_for_review": "Items selected for review",
})
display(qa12_detail.round(6))
export_table(
    qa12_detail,
    "table_original_qa12_consistency",
    "QA1.2 recurring-component consistency by target language and component class.",
    "tab:original-qa12-consistency",
)

qa12_variable.to_csv(
    DATA_DIR / "qa12_variable_mappings_for_review.csv",
    index=False,
)


### 3.2 QA2 knowledge preservation and target RDF--text faithfulness


In [ ]:
qa2_detail_columns = [
    "language_label",
    "lexicalisations",
    "records",
    "text_similarity_mean",
    "triple_similarity_mean",
    "predicate_similarity_mean",
    "structured_predicate_review_rate",
    "source_target_literal_preservation_mean",
    "source_target_full_literal_preservation_rate",
    "minimum_triple_coverage_mean",
    "coverage_pass_rate",
    "minimum_text_groundedness_mean",
    "groundedness_pass_rate",
    "target_text_literal_retention_mean",
    "target_text_full_literal_retention_rate",
    "relative_literal_retention",
]
require_columns(qa2_summary, qa2_detail_columns, "QA2 detailed summary")

qa2_detail = language_reindex(qa2_summary.copy())[
    qa2_detail_columns
].rename(columns={
    "language_label": "Language",
    "lexicalisations": "Lexicalisations",
    "records": "RDF entries",
    "text_similarity_mean": "TextSim",
    "triple_similarity_mean": "TripleSim",
    "predicate_similarity_mean": "PredSim",
    "structured_predicate_review_rate": "Unchanged-predicate review rate",
    "source_target_literal_preservation_mean": "Source-to-target literal mean",
    "source_target_full_literal_preservation_rate": "Full source literal rate",
    "minimum_triple_coverage_mean": "Minimum coverage mean",
    "coverage_pass_rate": "Coverage pass rate",
    "minimum_text_groundedness_mean": "Minimum groundedness mean",
    "groundedness_pass_rate": "Groundedness pass rate",
    "target_text_literal_retention_mean": "Target literal mean",
    "target_text_full_literal_retention_rate": "Full target literal rate",
    "relative_literal_retention": "Relative literal retention",
})
display(qa2_detail.round(6))
export_table(
    qa2_detail,
    "table_original_qa2_details",
    "QA2 source-preservation and target RDF--text faithfulness indicators on the original resources.",
    "tab:original-qa2-details",
)

qa2_thresholds.to_csv(
    DATA_DIR / "qa22_english_calibration_thresholds.csv",
    index=False,
)
qa2_bootstrap.to_csv(
    DATA_DIR / "qa2_spanish_catalan_cluster_bootstrap.csv",
    index=False,
)


In [ ]:
# QA2 review localisation uses only the final defined review decisions and
# exact literal failures. Continuous QA2.1 similarities are not thresholded.

qa2_localisation = qa2_metrics.copy()

source_predicate_priority = coerce_boolean(
    qa2_localisation["structured_predicate_review_flag"]
)
source_literal = pd.to_numeric(
    qa2_localisation["source_target_literal_preservation"],
    errors="coerce",
)
source_literal_priority = source_literal.notna() & source_literal.lt(
    1.0 - 1e-12
)
qa2_localisation["qa21_review_priority"] = (
    source_predicate_priority | source_literal_priority
)

coverage_pass = qa2_localisation[
    "english_calibrated_coverage_pass"
]
groundedness_pass = qa2_localisation[
    "english_calibrated_groundedness_pass"
]
coverage_failure = coverage_pass.notna() & ~coerce_boolean(coverage_pass)
groundedness_failure = (
    groundedness_pass.notna()
    & ~coerce_boolean(groundedness_pass)
)
target_literal = pd.to_numeric(
    qa2_localisation["target_text_literal_retention"],
    errors="coerce",
)
target_literal_priority = target_literal.notna() & target_literal.lt(
    1.0 - 1e-12
)
qa2_localisation["qa22_review_priority"] = (
    coverage_failure
    | groundedness_failure
    | target_literal_priority
)

qa2_localisation["Review localisation"] = np.select(
    [
        ~qa2_localisation["qa21_review_priority"]
        & ~qa2_localisation["qa22_review_priority"],
        qa2_localisation["qa21_review_priority"]
        & ~qa2_localisation["qa22_review_priority"],
        ~qa2_localisation["qa21_review_priority"]
        & qa2_localisation["qa22_review_priority"],
        qa2_localisation["qa21_review_priority"]
        & qa2_localisation["qa22_review_priority"],
    ],
    [
        "No defined priority",
        "QA2.1 only",
        "QA2.2 only",
        "Both QA2.1 and QA2.2",
    ],
    default="Unclassified",
)

localisation_order = [
    "No defined priority",
    "QA2.1 only",
    "QA2.2 only",
    "Both QA2.1 and QA2.2",
]
qa2_localisation["Review localisation"] = pd.Categorical(
    qa2_localisation["Review localisation"],
    categories=localisation_order,
    ordered=True,
)
qa2_localisation_summary = (
    qa2_localisation.groupby(
        ["language_label", "Review localisation"],
        observed=False,
    )
    .size()
    .rename("Lexicalisations")
    .reset_index()
)
totals = qa2_localisation_summary.groupby(
    "language_label"
)["Lexicalisations"].transform("sum")
qa2_localisation_summary["Rate"] = (
    qa2_localisation_summary["Lexicalisations"] / totals
)
qa2_localisation_summary = qa2_localisation_summary.sort_values(
    ["language_label", "Review localisation"]
)

display(qa2_localisation_summary)
export_table(
    qa2_localisation_summary.rename(columns={
        "language_label": "Language",
    }),
    "table_original_qa2_review_localisation",
    "Localisation of final QA2 review priorities in the original resources.",
    "tab:original-qa2-review-localisation",
)

localisation_plot = (
    qa2_localisation_summary.pivot_table(
        index="language_label",
        columns="Review localisation",
        values="Rate",
        aggfunc="first",
        observed=False,
    )
    .reindex(index=LANGUAGE_ORDER, columns=localisation_order)
    .fillna(0.0)
)
fig, ax = plt.subplots(figsize=(7.4, 4.5))
localisation_plot.plot(kind="bar", stacked=True, ax=ax, rot=0)
ax.set_ylim(0, 1)
ax.set_ylabel("Proportion of lexicalisations")
ax.set_xlabel("")
ax.set_title("QA2 review-priority localisation on original resources")
ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
save_figure(fig, "fig_original_qa2_review_localisation")


In [ ]:
qa2_metric_plot = qa2_summary.set_index("language_label")[
    [
        "text_similarity_mean",
        "triple_similarity_mean",
        "predicate_similarity_mean",
        "minimum_triple_coverage_mean",
        "minimum_text_groundedness_mean",
        "source_target_literal_preservation_mean",
        "target_text_literal_retention_mean",
    ]
].rename(columns={
    "text_similarity_mean": "TextSim",
    "triple_similarity_mean": "TripleSim",
    "predicate_similarity_mean": "PredSim",
    "minimum_triple_coverage_mean": "Minimum coverage",
    "minimum_text_groundedness_mean": "Minimum groundedness",
    "source_target_literal_preservation_mean": "Source literal",
    "target_text_literal_retention_mean": "Target literal",
}).T.reindex(columns=LANGUAGE_ORDER)

fig, ax = plt.subplots(figsize=(8.3, 4.8))
qa2_metric_plot.plot(kind="bar", ax=ax, rot=25)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Mean score")
ax.set_xlabel("")
ax.set_title("QA2 indicators on the original resources")
ax.legend(title="")
save_figure(fig, "fig_original_qa2_metric_profile")


### 3.3 QA3 target-language validity and legitimate variation


In [ ]:
qa3_language_detail = language_reindex(
    qa3_language_summary.copy()
)[
    [
        "language_label",
        "variants",
        "hard_language_flag_rate",
        "valid_target_language_rate",
        "wrong_language_rate",
        "exact_source_copy_rate",
        "code_switch_rate",
        "mean_target_language_probability",
    ]
].rename(columns={
    "language_label": "Language",
    "variants": "Lexicalisations",
    "hard_language_flag_rate": "Hard-language flag rate",
    "valid_target_language_rate": "Valid-language rate",
    "wrong_language_rate": "Wrong-language rate",
    "exact_source_copy_rate": "Exact English-copy rate",
    "code_switch_rate": "Code-switch rate",
    "mean_target_language_probability": "Mean target-language probability",
})
display(qa3_language_detail.round(6))
export_table(
    qa3_language_detail,
    "table_original_qa31_details",
    "QA3.1 target-language validity and source-language leakage indicators.",
    "tab:original-qa31-details",
)

qa3_variation_detail = language_reindex(
    qa3_variation_summary.copy()
)[
    [
        "language_label",
        "mean_expansion_ratio",
        "median_expansion_ratio",
        "expansion_ratio_sd",
        "aligned_reference_eligible",
        "mean_aligned_reference_advantage",
        "positive_aligned_reference_advantage_rate",
    ]
].rename(columns={
    "language_label": "Language",
    "mean_expansion_ratio": "Mean expansion ratio",
    "median_expansion_ratio": "Median expansion ratio",
    "expansion_ratio_sd": "Expansion-ratio SD",
    "aligned_reference_eligible": "Eligible aligned-reference comparisons",
    "mean_aligned_reference_advantage": "Mean aligned-reference advantage",
    "positive_aligned_reference_advantage_rate": "Positive-advantage rate",
})
display(qa3_variation_detail.round(6))
export_table(
    qa3_variation_detail,
    "table_original_qa32_details",
    "QA3.2 descriptive variation and aligned-reference advantage indicators.",
    "tab:original-qa32-details",
)


In [ ]:
qa3_status = qa3_language_summary[
    [
        "language_label",
        "valid_target_language_rate",
        "wrong_language_rate",
    ]
].copy()
qa3_status["Uncertain"] = (
    1.0
    - qa3_status["valid_target_language_rate"]
    - qa3_status["wrong_language_rate"]
).clip(lower=0.0)
qa3_status = qa3_status.rename(columns={
    "valid_target_language_rate": "Valid",
    "wrong_language_rate": "Wrong language",
}).set_index("language_label")[
    ["Valid", "Uncertain", "Wrong language"]
].reindex(LANGUAGE_ORDER)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
qa3_status.plot(kind="bar", stacked=True, ax=ax, rot=0)
ax.set_ylim(0, 1)
ax.set_ylabel("Proportion of lexicalisations")
ax.set_xlabel("")
ax.set_title("QA3.1 whole-text language-identification status")
ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
save_figure(fig, "fig_original_qa31_language_status")

qa3_leakage = qa3_language_summary.set_index("language_label")[
    [
        "wrong_language_rate",
        "exact_source_copy_rate",
        "code_switch_rate",
    ]
].rename(columns={
    "wrong_language_rate": "Wrong language",
    "exact_source_copy_rate": "Exact English copy",
    "code_switch_rate": "English code switch",
}).T.reindex(columns=LANGUAGE_ORDER) * 100.0

fig, ax = plt.subplots(figsize=(7.3, 4.4))
qa3_leakage.plot(kind="bar", ax=ax, rot=20)
ax.set_ylabel("Percentage of lexicalisations")
ax.set_xlabel("")
ax.set_title("QA3.1 high-confidence language-risk rates")
ax.legend(title="")
save_figure(fig, "fig_original_qa31_language_risks")


In [ ]:
expansion_groups = [
    pd.to_numeric(
        qa3_metrics.loc[
            qa3_metrics["language_label"].eq(language),
            "expansion_ratio",
        ],
        errors="coerce",
    ).dropna()
    for language in LANGUAGE_ORDER
]
fig, ax = plt.subplots(figsize=(6.8, 4.4))
ax.boxplot(expansion_groups, labels=LANGUAGE_ORDER, showfliers=False)
ax.axhline(1.0, linewidth=1)
ax.set_ylabel("Target / English token count")
ax.set_title("QA3.2 expansion-ratio distribution")
save_figure(fig, "fig_original_qa32_expansion_ratio")

advantage_groups = [
    pd.to_numeric(
        qa3_metrics.loc[
            qa3_metrics["language_label"].eq(language),
            "aligned_reference_advantage",
        ],
        errors="coerce",
    ).dropna()
    for language in LANGUAGE_ORDER
]
if any(len(values) for values in advantage_groups):
    fig, ax = plt.subplots(figsize=(6.8, 4.4))
    ax.boxplot(
        advantage_groups,
        labels=LANGUAGE_ORDER,
        showfliers=False,
    )
    ax.axhline(0.0, linewidth=1)
    ax.set_ylabel("Aligned-reference advantage")
    ax.set_title("QA3.2 aligned-reference advantage on original resources")
    save_figure(fig, "fig_original_qa32_aligned_reference_advantage")
else:
    print(
        "No original-resource entries have an eligible alternative English "
        "reference; the aligned-reference-advantage distribution was skipped."
    )


## 4. Controlled framework-validation results


In [ ]:
stress_overall = stress_combined[
    ~stress_combined["corruption_type"].isin(
        ["clean", "benign_alternative_reference"]
    )
].copy()

summary_rows = []
for (dimension, language), group in stress_overall.groupby(
    ["audit_dimension", "language_label"],
    sort=True,
):
    weights = pd.to_numeric(group["variants"], errors="coerce").to_numpy(
        dtype=float
    )
    rates = pd.to_numeric(
        group["primary_detection_rate"],
        errors="coerce",
    ).to_numpy(dtype=float)
    valid = np.isfinite(weights) & np.isfinite(rates) & (weights > 0)
    if not valid.any():
        continue
    summary_rows.append({
        "Audit dimension": dimension,
        "Language": language,
        "Controlled variants": int(weights[valid].sum()),
        "Corruption types": int(group["corruption_type"].nunique()),
        "Weighted detection rate": float(
            np.average(rates[valid], weights=weights[valid])
        ),
        "Minimum detection rate": float(
            np.nanmin(rates[valid])
        ),
    })

dimension_order = ["QA1.1", "QA1.2", "QA2", "QA3.1"]
stress_overall_table = pd.DataFrame(summary_rows)
stress_overall_table["Audit dimension"] = pd.Categorical(
    stress_overall_table["Audit dimension"],
    categories=dimension_order,
    ordered=True,
)
stress_overall_table["Language"] = pd.Categorical(
    stress_overall_table["Language"],
    categories=LANGUAGE_ORDER,
    ordered=True,
)
stress_overall_table = stress_overall_table.sort_values(
    ["Audit dimension", "Language"]
).reset_index(drop=True)

display(stress_overall_table.round(6))
export_table(
    stress_overall_table,
    "table_stress_overall_summary",
    "Aggregate controlled stress-test performance under the final metric definitions.",
    "tab:stress-test-summary",
)


In [ ]:
def plot_detection_heatmap(
    data: pd.DataFrame,
    dimensions: list[str],
    title: str,
    stem: str,
    label_width: int,
) -> None:
    panel = data[data["audit_dimension"].isin(dimensions)].copy()
    if panel.empty:
        print(f"No data for {dimensions}; skipped {stem}.")
        return

    panel["Corruption"] = panel["corruption_type"].map(pretty_corruption)
    order = panel[
        ["audit_dimension", "Corruption"]
    ].drop_duplicates()
    matrix = panel.pivot_table(
        index="language_label",
        columns=["audit_dimension", "Corruption"],
        values="primary_detection_rate",
        aggfunc="mean",
    ).reindex(index=LANGUAGE_ORDER)
    matrix = matrix.reindex(
        columns=pd.MultiIndex.from_frame(order)
    )

    show_dimension = len(dimensions) > 1
    matrix.columns = [
        fill(
            (
                f"{dimension}: {corruption}"
                if show_dimension
                else corruption
            ),
            width=label_width,
            break_long_words=False,
            break_on_hyphens=False,
        )
        for dimension, corruption in matrix.columns
    ]

    width = max(7.2, min(11.0, 0.78 * len(matrix.columns) + 3.0))
    fig, ax = plt.subplots(figsize=(width, 3.7))
    image = ax.imshow(
        matrix.to_numpy(dtype=float),
        aspect="auto",
        vmin=0,
        vmax=1,
    )
    ax.set_xticks(range(len(matrix.columns)), matrix.columns)
    ax.tick_params(axis="x", rotation=38)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_title(title)
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix.iloc[row, column]
            if pd.notna(value):
                ax.text(
                    column,
                    row,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                )
    fig.colorbar(image, ax=ax, label="Detection rate")
    save_figure(fig, stem)

plot_detection_heatmap(
    stress_overall,
    ["QA1.1", "QA1.2"],
    "QA1: Structural and consistency corruption detection",
    "fig_qa1_detection_heatmap",
    18,
)
plot_detection_heatmap(
    stress_overall,
    ["QA2"],
    "QA2: Semantic and factual corruption detection",
    "fig_qa2_detection_heatmap",
    17,
)
plot_detection_heatmap(
    stress_overall,
    ["QA3.1"],
    "QA3.1: Target-language corruption detection",
    "fig_qa3_detection_heatmap",
    22,
)


### 4.1 QA2 controlled semantic and factual perturbations


In [ ]:
qa2_stress_table = stress_qa2.copy()
qa2_stress_table["Corruption"] = qa2_stress_table[
    "corruption_type"
].map(pretty_corruption)
qa2_stress_table["Metric"] = qa2_stress_table[
    "primary_metric"
].map(pretty_metric)
qa2_stress_table = qa2_stress_table[
    [
        "language_label",
        "Corruption",
        "Metric",
        "severity",
        "variants",
        "expected_direction_rate",
        "mean_delta",
        "cluster_bootstrap_ci_low",
        "cluster_bootstrap_ci_high",
        "auroc_clean_vs_variant",
        "auprc_clean_vs_variant",
    ]
].rename(columns={
    "language_label": "Language",
    "severity": "Severity",
    "variants": "Variants",
    "expected_direction_rate": "Detection rate",
    "mean_delta": "Mean paired change",
    "cluster_bootstrap_ci_low": "CI low",
    "cluster_bootstrap_ci_high": "CI high",
    "auroc_clean_vs_variant": "AUROC",
    "auprc_clean_vs_variant": "AUPRC",
}).sort_values(["Corruption", "Language"])

display(qa2_stress_table.round(6))
export_table(
    qa2_stress_table,
    "table_stress_qa2_corruption_results",
    "QA2 response to controlled semantic and factual perturbations.",
    "tab:stress-qa2-corruption",
)

forest = stress_qa2[
    stress_qa2["cluster_bootstrap_ci_low"].notna()
    & stress_qa2["cluster_bootstrap_ci_high"].notna()
].copy()
forest["Corruption"] = forest["corruption_type"].map(pretty_corruption)
corruption_order = (
    forest.groupby("Corruption")["mean_delta"]
    .mean()
    .sort_values()
    .index.tolist()
)
positions = np.arange(len(corruption_order), dtype=float)
offsets = {"Spanish": -0.14, "Catalan": 0.14}

fig, ax = plt.subplots(
    figsize=(8.8, max(5.0, 0.48 * len(corruption_order) + 1.6))
)
for language in LANGUAGE_ORDER:
    subset = (
        forest[forest["language_label"].eq(language)]
        .set_index("Corruption")
        .reindex(corruption_order)
    )
    x = pd.to_numeric(subset["mean_delta"], errors="coerce").to_numpy()
    low = pd.to_numeric(
        subset["cluster_bootstrap_ci_low"], errors="coerce"
    ).to_numpy()
    high = pd.to_numeric(
        subset["cluster_bootstrap_ci_high"], errors="coerce"
    ).to_numpy()
    lower_error = x - low
    upper_error = high - x
    ax.errorbar(
        x,
        positions + offsets[language],
        xerr=np.vstack([lower_error, upper_error]),
        fmt="o",
        capsize=3,
        label=language,
    )
ax.axvline(0.0, linewidth=1)
ax.set_yticks(positions, corruption_order)
ax.set_xlabel("Mean paired score change (corrupted − clean)")
ax.set_title("QA2 paired metric response with record-cluster intervals")
ax.legend(title="")
save_figure(fig, "fig_qa2_effect_sizes_forest")


In [ ]:
if not stress_qa2_omission.empty:
    omission_long = stress_qa2_omission.melt(
        id_vars=["language_label", "paired_instances"],
        value_vars=[
            "severe_lower_coverage_rate",
            "severe_lower_text_similarity_rate",
        ],
        var_name="Measure",
        value_name="Rate",
    )
    omission_long["Measure"] = omission_long["Measure"].map({
        "severe_lower_coverage_rate": "Minimum triple coverage",
        "severe_lower_text_similarity_rate": "Text similarity",
    })
    omission_matrix = omission_long.pivot_table(
        index="Measure",
        columns="language_label",
        values="Rate",
    ).reindex(columns=LANGUAGE_ORDER)
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    omission_matrix.plot(kind="bar", ax=ax, rot=0)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Rate: severe score below mild score")
    ax.set_xlabel("")
    ax.set_title("QA2 nested-omission severity ordering")
    ax.legend(title="")
    save_figure(fig, "fig_qa2_omission_severity")

if not stress_qa2_addition.empty:
    addition = stress_qa2_addition.set_index(
        "language_label"
    ).reindex(LANGUAGE_ORDER)
    fig, ax = plt.subplots(figsize=(6.7, 4.3))
    ax.bar(
        addition.index,
        addition["unrelated_lower_groundedness_rate"],
    )
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Rate: unrelated addition less grounded")
    ax.set_title("QA2 unsupported-addition severity ordering")
    save_figure(fig, "fig_qa2_addition_severity")


### 4.2 QA3 language corruption and benign variation


In [ ]:
qa3_control_table = stress_qa3_controls.rename(columns={
    "qa_track": "Control",
    "language_label": "Language",
    "variants": "Variants",
    "hard_flag_false_positive_rate": "Hard false-positive rate",
    "valid_target_language_rate": "Valid target-language rate",
    "wrong_language_rate": "Wrong-language rate",
    "exact_copy_rate": "Exact-copy rate",
    "code_switch_rate": "Code-switch rate",
}).copy()
qa3_control_table["Control"] = qa3_control_table["Control"].replace({
    "CLEAN": "Clean",
    "BENIGN-VAR": "Benign variation",
})
display(qa3_control_table.round(6))
export_table(
    qa3_control_table,
    "table_stress_qa3_control_selectivity",
    "QA3.1 language-diagnostic activation rates on clean and benign controls.",
    "tab:stress-qa3-controls",
)

qa3_sensitivity = stress_qa3[
    ["language_label", "corruption_type", "detection_rate"]
].copy()
qa3_sensitivity["Condition"] = qa3_sensitivity[
    "corruption_type"
].map(pretty_corruption)
qa3_sensitivity = qa3_sensitivity.rename(columns={
    "language_label": "Language",
    "detection_rate": "Rate",
})[["Condition", "Language", "Rate"]]

specificity_rows = []
for row in stress_qa3_controls.itertuples(index=False):
    control_label = {
        "CLEAN": "Clean specificity",
        "BENIGN-VAR": "Benign-variation specificity",
    }.get(row.qa_track, f"{row.qa_track} specificity")
    specificity_rows.append({
        "Condition": control_label,
        "Language": row.language_label,
        "Rate": 1.0 - float(row.hard_flag_false_positive_rate),
    })

qa3_balance = pd.concat(
    [qa3_sensitivity, pd.DataFrame(specificity_rows)],
    ignore_index=True,
)
condition_order = [
    "Full English copy",
    "English-clause insertion",
    "Clean specificity",
    "Benign-variation specificity",
]
qa3_balance_matrix = qa3_balance.pivot_table(
    index="Condition",
    columns="Language",
    values="Rate",
).reindex(index=condition_order, columns=LANGUAGE_ORDER)

fig, ax = plt.subplots(figsize=(7.7, 4.7))
qa3_balance_matrix.plot(kind="bar", ax=ax, rot=18)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Rate")
ax.set_xlabel("")
ax.set_title("QA3.1 sensitivity and control specificity")
ax.legend(title="")
save_figure(fig, "fig_qa3_sensitivity_selectivity")


In [ ]:
if not stress_qa3_benign.empty:
    qa32_benign_table = stress_qa3_benign.rename(columns={
        "language_label": "Language",
        "variants": "Variants",
        "mean_expansion_change": "Mean expansion-ratio change",
        "mean_aligned_reference_advantage": "Mean aligned-reference advantage",
        "positive_aligned_reference_advantage_rate": "Positive-advantage rate",
    })[
        [
            "Language",
            "Variants",
            "Mean expansion-ratio change",
            "Mean aligned-reference advantage",
            "Positive-advantage rate",
        ]
    ]
    display(qa32_benign_table.round(6))
    export_table(
        qa32_benign_table,
        "table_stress_qa32_benign_variation",
        "QA3.2 aligned-reference evidence on benign alternative lexicalisations.",
        "tab:stress-qa32-benign",
    )

    positive_advantage = qa32_benign_table.set_index(
        "Language"
    )["Positive-advantage rate"].reindex(LANGUAGE_ORDER)
    fig, ax = plt.subplots(figsize=(6.6, 4.2))
    ax.bar(positive_advantage.index, positive_advantage.values)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Positive aligned-reference advantage rate")
    ax.set_title("QA3.2 selectivity on benign alternative references")
    save_figure(fig, "fig_qa32_benign_aligned_reference")


## 5. Reproducibility manifest and recommended chapter selection


In [ ]:
key_results = {
    "metric_core_expected_compatibility": EXPECTED_CORE_COMPATIBILITY,
    "original_resource_inventory": inventory.to_dict(orient="records"),
    "original_audit_summary": original_overview.to_dict(orient="records"),
    "qa2_review_localisation": (
        qa2_localisation_summary.assign(
            **{
                "Review localisation": qa2_localisation_summary[
                    "Review localisation"
                ].astype(str)
            }
        ).to_dict(orient="records")
    ),
    "stress_overall_summary": (
        stress_overall_table.assign(
            **{
                "Audit dimension": stress_overall_table[
                    "Audit dimension"
                ].astype(str),
                "Language": stress_overall_table[
                    "Language"
                ].astype(str),
            }
        ).to_dict(orient="records")
    ),
    "qa3_benign_variation": (
        stress_qa3_benign.to_dict(orient="records")
        if not stress_qa3_benign.empty
        else []
    ),
    "interpretation_notes": {
        "qa12": (
            "Weighted dominant mapping rate is reported separately for "
            "language and component class."
        ),
        "qa2_review_localisation": (
            "QA2.1 priority uses the structured unchanged-predicate review "
            "flag or incomplete source-to-target literal preservation. "
            "QA2.2 priority uses calibrated coverage failure, calibrated "
            "groundedness failure, or incomplete target literal retention. "
            "Continuous QA2.1 similarities are not thresholded."
        ),
        "qa32": (
            "QA3.2 is descriptive and is excluded from aggregate "
            "corruption-detection rates."
        ),
    },
}
(DATA_DIR / "chapter_key_results.json").write_text(
    json.dumps(key_results, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

def lookup(frame: pd.DataFrame, language: str, column: str) -> float:
    return float(
        frame.loc[
            frame["language_label"].eq(language),
            column,
        ].iloc[0]
    )

macro_values = {
    "OrigEsRDFEntries": int(
        inventory.loc[
            inventory["Language"].eq("Spanish"),
            "RDF entries",
        ].iloc[0]
    ),
    "OrigCaRDFEntries": int(
        inventory.loc[
            inventory["Language"].eq("Catalan"),
            "RDF entries",
        ].iloc[0]
    ),
    "OrigEsLexicalisations": int(
        inventory.loc[
            inventory["Language"].eq("Spanish"),
            "Aligned lexicalisations",
        ].iloc[0]
    ),
    "OrigCaLexicalisations": int(
        inventory.loc[
            inventory["Language"].eq("Catalan"),
            "Aligned lexicalisations",
        ].iloc[0]
    ),
    "OrigEsRecordIntegrity": lookup(
        qa11_summary, "Spanish", "integrity_pass_rate"
    ),
    "OrigCaRecordIntegrity": lookup(
        qa11_summary, "Catalan", "integrity_pass_rate"
    ),
    "OrigEsTextSim": lookup(
        qa2_summary, "Spanish", "text_similarity_mean"
    ),
    "OrigCaTextSim": lookup(
        qa2_summary, "Catalan", "text_similarity_mean"
    ),
    "OrigEsCoveragePass": lookup(
        qa2_summary, "Spanish", "coverage_pass_rate"
    ),
    "OrigCaCoveragePass": lookup(
        qa2_summary, "Catalan", "coverage_pass_rate"
    ),
    "OrigEsGroundednessPass": lookup(
        qa2_summary, "Spanish", "groundedness_pass_rate"
    ),
    "OrigCaGroundednessPass": lookup(
        qa2_summary, "Catalan", "groundedness_pass_rate"
    ),
    "OrigEsRelativeLiteralRetention": lookup(
        qa2_summary, "Spanish", "relative_literal_retention"
    ),
    "OrigCaRelativeLiteralRetention": lookup(
        qa2_summary, "Catalan", "relative_literal_retention"
    ),
    "OrigEsValidLanguage": lookup(
        qa3_language_summary, "Spanish", "valid_target_language_rate"
    ),
    "OrigCaValidLanguage": lookup(
        qa3_language_summary, "Catalan", "valid_target_language_rate"
    ),
    "OrigEsCodeSwitch": lookup(
        qa3_language_summary, "Spanish", "code_switch_rate"
    ),
    "OrigCaCodeSwitch": lookup(
        qa3_language_summary, "Catalan", "code_switch_rate"
    ),
    "OrigEsExpansionRatio": lookup(
        qa3_variation_summary, "Spanish", "mean_expansion_ratio"
    ),
    "OrigCaExpansionRatio": lookup(
        qa3_variation_summary, "Catalan", "mean_expansion_ratio"
    ),
}

macro_lines = [
    "% Automatically generated by the Chapter 5 audit summary notebook."
]
for name, value in macro_values.items():
    if isinstance(value, int):
        rendered = f"{value:,}"
    else:
        rendered = f"{float(value):.3f}"
    macro_lines.append(
        rf"\newcommand{{\{name}}}{{{rendered}}}"
    )
(TABLE_DIR / "chapter_audit_result_macros.tex").write_text(
    "\n".join(macro_lines) + "\n",
    encoding="utf-8",
)

input_paths = [
    QA1_RESULTS_DIR / "qa11_original_structural_summary.csv",
    QA1_RESULTS_DIR / "qa12_original_consistency_summary.csv",
    QA1_RESULTS_DIR / "qa12_variable_mappings.csv",
    QA1_RESULTS_DIR / "qa1_entry_inventory.csv",
    QA1_RESULTS_DIR / "qa1_run_metadata.json",
    QA2_RESULTS_DIR / "qa2_original_summary.csv",
    (
        QA2_RESULTS_DIR / "qa2_lexicalisation_metrics.pkl"
        if (QA2_RESULTS_DIR / "qa2_lexicalisation_metrics.pkl").exists()
        else QA2_RESULTS_DIR / "qa2_lexicalisation_metrics.csv"
    ),
    QA2_RESULTS_DIR / "qa22_english_calibration_thresholds.csv",
    QA2_RESULTS_DIR / "qa2_spanish_catalan_cluster_bootstrap.csv",
    QA2_RESULTS_DIR / "qa2_run_metadata.json",
    QA3_RESULTS_DIR / "qa3_original_language_summary.csv",
    QA3_RESULTS_DIR / "qa3_variation_summary.csv",
    (
        QA3_RESULTS_DIR / "qa3_lexicalisation_metrics.pkl"
        if (QA3_RESULTS_DIR / "qa3_lexicalisation_metrics.pkl").exists()
        else QA3_RESULTS_DIR / "qa3_lexicalisation_metrics.csv"
    ),
    QA3_RESULTS_DIR / "qa3_run_metadata.json",
    STRESS_RESULTS_DIR / "combined_detection_summary.csv",
    STRESS_RESULTS_DIR / "qa1_structural_detection_summary.csv",
    STRESS_RESULTS_DIR / "qa1_consistency_selectivity_summary.csv",
    STRESS_RESULTS_DIR / "qa2_corruption_detection_summary.csv",
    STRESS_RESULTS_DIR / "qa2_benign_variation_summary.csv",
    STRESS_RESULTS_DIR / "qa2_omission_severity_summary.csv",
    STRESS_RESULTS_DIR / "qa2_addition_severity_summary.csv",
    STRESS_RESULTS_DIR / "qa3_corruption_detection_summary.csv",
    STRESS_RESULTS_DIR / "qa3_control_selectivity_summary.csv",
    STRESS_RESULTS_DIR / "qa3_benign_variation_summary.csv",
    STRESS_RESULTS_DIR / "ControlledStress_run_metadata.json",
]

recommended_main = {
    "tables": [
        "table_original_resource_inventory.tex",
        "table_original_audit_summary.tex",
        "table_stress_overall_summary.tex",
        "table_stress_qa32_benign_variation.tex",
    ],
    "figures": [
        "fig_original_qa2_review_localisation.pdf",
        "fig_qa1_detection_heatmap.pdf",
        "fig_qa2_detection_heatmap.pdf",
        "fig_qa2_effect_sizes_forest.pdf",
        "fig_qa3_detection_heatmap.pdf",
        "fig_qa3_sensitivity_selectivity.pdf",
        "fig_qa32_benign_aligned_reference.pdf",
    ],
}
recommended_supplementary = {
    "tables": sorted(
        path.name
        for path in TABLE_DIR.glob("*.tex")
        if path.name not in recommended_main["tables"]
        and path.name != "chapter_audit_result_macros.tex"
    ),
    "figures": sorted(
        path.name
        for path in FIGURE_DIR.glob("*.pdf")
        if path.name not in recommended_main["figures"]
    ),
}

manifest = {
    "expected_metric_core_compatibility": EXPECTED_CORE_COMPATIBILITY,
    "source_result_directories": {
        "qa1": str(QA1_RESULTS_DIR),
        "qa2": str(QA2_RESULTS_DIR),
        "qa3": str(QA3_RESULTS_DIR),
        "ControlledStress": str(STRESS_RESULTS_DIR),
    },
    "export_directory": str(EXPORT_DIR),
    "input_files": [file_fingerprint(path) for path in input_paths],
    "recommended_main_chapter_outputs": recommended_main,
    "recommended_supplementary_outputs": recommended_supplementary,
    "all_pdf_figures": sorted(
        path.name for path in FIGURE_DIR.glob("*.pdf")
    ),
    "all_png_figures": sorted(
        path.name for path in FIGURE_DIR.glob("*.png")
    ),
    "all_latex_tables": sorted(
        path.name for path in TABLE_DIR.glob("*.tex")
    ),
    "all_csv_tables": sorted(
        path.name for path in TABLE_DIR.glob("*.csv")
    ),
}
(EXPORT_DIR / "chapter_audit_output_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

notes = (
    "Chapter 5 audit-results export\n\n"
    "The outputs in this directory were reconstructed from the final QA1, "
    "QA2, QA3, and ControlledStress result files. No metric inference was "
    "repeated here.\n\n"
    "Important interpretation rules:\n"
    "1. QA1.2 remains stratified by language and component class.\n"
    "2. QA2 review localisation uses only the final defined review decisions "
    "and exact literal failures; continuous QA2.1 similarities are not "
    "thresholded.\n"
    "3. QA3.2 is descriptive and is excluded from weighted corruption "
    "detection.\n"
    "4. Manual-validation results are not inferred by this notebook.\n\n"
    "Use chapter_audit_output_manifest.json to identify the recommended "
    "main-text and supplementary outputs.\n"
)
(EXPORT_DIR / "README.txt").write_text(notes, encoding="utf-8")

print("Exported figures:", len(manifest["all_pdf_figures"]))
print("Exported LaTeX tables:", len(manifest["all_latex_tables"]))
print("Manifest:", EXPORT_DIR / "chapter_audit_output_manifest.json")


In [ ]:
selection_rows = []
for kind, filenames in recommended_main.items():
    for filename in filenames:
        selection_rows.append({
            "Placement": "Main chapter",
            "Type": kind[:-1].title(),
            "File": filename,
        })
for kind, filenames in recommended_supplementary.items():
    for filename in filenames:
        selection_rows.append({
            "Placement": "Supplementary / optional",
            "Type": kind[:-1].title(),
            "File": filename,
        })

recommended_selection = pd.DataFrame(selection_rows)
display(recommended_selection)
recommended_selection.to_csv(
    DATA_DIR / "recommended_chapter_selection.csv",
    index=False,
)
